In [ ]:
from openai import OpenAI
from presidio_analyzer import AnalyzerEngine
from presidio_anonymizer import AnonymizerEngine
from presidio_anonymizer.entities import OperatorConfig

import ollama
import os
import logging
import time

### Building an AI Privacy Router

This is just a rough example of some concepts I've been working out on [building an AI Privacy router](https://blog.kjamistan.com/building-an-ai-privacy-router.html). It's here to give you some ways to grapple and think about routing requests and prompts based on their sensitivity, not as a "directly copy this and implement it in your workflow." 

Please do not use this in a real production system, again, it's a toy example to spark discussion and design!

Want to explore these topics in depth? Join my first cohort of [Practical AI Privacy](https://maven.com/katharine-jarmul/practical-ai-privacy) on April 20-May 30, 2026. Hope to see you there!

In [ ]:
ollama.pull('mistral')
ollama.pull('llama-guard3')

In [ ]:
# Note: please don't actually use logging like this ! It's an example to illustrate how you might audit your router

logger = logging.getLogger()
logger.setLevel(logging.DEBUG)

In [ ]:
def get_guardrail_response(prompt):
    start = time.time()
    ollama_client = ollama.Client()
    response = ollama_client.chat(
        model='llama-guard3',
        messages=[
            {'role': 'user', 
             'content': prompt}]
    ).message.content
    elapsed = time.time() - start
    if 'unsafe' in response:
        logging.debug(f"Guardrail flags: {response}")
    logging.debug(f"Guardrail Model took {elapsed}s")
    return response

In [ ]:
def is_sensitive(text):
    return 'S7' in get_guardrail_response(text)

In [ ]:
def generate_aes_key(bit_length=256): 
    byte_length = int(bit_length / 8)
    return os.urandom(byte_length)

In [ ]:
AES_KEY = generate_aes_key()

In [ ]:
entities_to_remove = [
    "PHONE_NUMBER",
    "US_DRIVER_LICENSE",
    "US_PASSPORT",
    "LOCATION",
    "CREDIT_CARD",
    "CRYPTO",
    "UK_NHS",
    "US_SSN",
    "US_BANK_NUMBER",
    "EMAIL_ADDRESS",
    "DATE_TIME",
    "IP_ADDRESS",
    "PERSON",
    "IBAN_CODE",
    "NRP",
    "US_ITIN",
    "MEDICAL_LICENSE",
    "URL"
]

In [ ]:
pseudonymization_operators = {
    "DEFAULT": OperatorConfig(
        "replace", 
        {"new_value": "<REDACTED>"}
    ),
    "PHONE_NUMBER": OperatorConfig(
        "mask", 
        {
            "type": "mask", 
            "masking_char" : "*",
            "chars_to_mask" : 12, 
            "from_end" : True
        }
    ),
    "US_PASSPORT": OperatorConfig(
        "encrypt", 
        {"key": AES_KEY}
    ),
}

In [ ]:
def pseudonymize_text(text, entities, operator_config):
    start = time.time()
    analyzer = AnalyzerEngine()
    analyzer_results = analyzer.analyze(
        text=text, 
        entities=entities, 
        language='en'
    )

    # Call the Pseudonymizer
    anonymizer = AnonymizerEngine()

    results = anonymizer.anonymize(
        text=text,
        analyzer_results=analyzer_results,    
        operators=operator_config
    )
    elapsed = time.time() - start
    logging.debug(f"Pseudonymization analysis and engine took {elapsed}s")
    return results.text

In [ ]:
def call_internal_service(prompt):
    start = time.time()
    client = ollama.Client()
    response = client.chat(
        model= "mistral",
        messages=[
            {'role': 'user', 
             'content': prompt}]
    ).message.content
    elapsed = time.time() - start
    logging.debug(f"Internal model response took {elapsed}s")
    return response   

In [ ]:
def call_external_service(prompt, original_model):
    start = time.time()
    client = OpenAI(
        base_url="http://192.168.1.176:7777/v1", # Change if you want to test with actual service
        api_key = "no-thank-u"                   # also change here :) 
    )
    response = client.chat.completions.create(
        model= original_model,
        messages=[
            {'role': 'user', 
             'content': prompt}]
    ).choices[0].message.content
    elapsed = time.time() - start
    logging.debug(f"External model response took {elapsed}s")
    return response  

In [ ]:
def route_requests(text, original_model, 
                   entities=entities_to_remove, 
                   operator_config=pseudonymization_operators):
    if is_sensitive(text):
        logging.debug('Text has privacy flags, entering routing logic')
        new_text = pseudonymize_text(text, entities, operator_config)
        print(new_text)
        if is_sensitive(new_text):
            logging.error('Pseudonymizer failed to protect input')
            response = call_internal_service(text)
    else:
        logging.debug(f"Text has no privacy flags, fwding to {original_model}")
        response = call_external_service(text, original_model)
    return response

In [ ]:
response = route_requests(
    "Please deliver me a sandwich to my house 123 Main Street. If you get lost call me at 888-222-3333",
    "qwen35_35"
)

In [ ]:
print(response)

In [ ]:
response = route_requests(
    "Can you tell me about quantum physics?",
    "qwen35_35"
)

In [ ]:
print(response)